# Step A — Two-Step Lasso Income Prediction
Replicates Section IV-B of Friedberg & Isaac (2024).

**Logic:**
1. **Step 1 (LassoCV LPM):** Predict P(incearn > 0). Convert to binary using the sample mean as threshold.
2. **Step 2 (LassoCV log-OLS):** Predict log(incearn) for positive earners only. Exponentiate to get levels.
3. **Final predicted income:** Step-2 level for predicted workers (Step 1 = 1), else 0.

Both steps are **estimated on 2012 only**, then **applied to 2012-2017**.

Features (paper footnote 22): 5-year age-group dummies, 4 education-level dummies,
number of children, race/sex/occupation/major/state dummies, and pairwise interactions.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

FINAL_PKL = "/Users/trinityrose/Desktop/acs_ssc_lasso_input.pkl"
OUT_PKL   = "/Users/trinityrose/Desktop/acs_ssc_predicted_v2.pkl"

df = pd.read_pickle(FINAL_PKL)
print(f"Loaded: {df.shape}  |  Years: {sorted(df['year'].unique())}")

Loaded: (37907, 51)  |  Years: [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017)]


In [3]:
# ── Fix missing sp_male ──────────────────────────────────────────
# build_data_V2.py omits sp_male from keep_cols.

if "sp_male" not in df.columns:
    df["sp_male"] = df["r_male"]
    print("Derived sp_male = r_male  (same-sex sample — sex is identical)")

# Sanity check: confirm all required feature columns are present
needed = [
    "r_agegroup", "r_edugroup", "r_race", "r_occbroad", "r_degbroad", "r_male",
    "sp_agegroup", "sp_edugroup", "sp_race", "sp_occbroad", "sp_degbroad", "sp_male",
    "c_children", "statefip", "r_incearn", "sp_incearn"
]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"Still missing columns: {missing}")
print("All required columns present.")

All required columns present.


In [4]:
# ── Feature builder ──────────────────────────────────────────────
def build_features(person_df: pd.DataFrame) -> pd.DataFrame:
    """
    Expects columns already renamed to the canonical names:
      r_agegroup, r_edugroup, r_race, r_occbroad, r_degbroad,
      r_male, statefip, c_children
    Returns dummy-expanded + pairwise-interaction matrix.
    """
    cat_cols = ["r_agegroup", "r_edugroup", "r_race",
                "r_occbroad", "r_degbroad", "statefip"]

    base = pd.get_dummies(
        person_df[cat_cols + ["r_male", "c_children"]],
        columns=cat_cols,
        drop_first=True,
        dtype=float
    )

    # Pairwise interactions — keep only non-trivial (sum >= 10)
    base_cols = list(base.columns)
    interact_parts = []
    for i, c1 in enumerate(base_cols):
        for c2 in base_cols[i + 1:]:
            s = base[c1] * base[c2]
            if s.sum() >= 10:
                interact_parts.append(s.rename(f"{c1}_x_{c2}"))

    if interact_parts:
        X = pd.concat([base, pd.concat(interact_parts, axis=1)], axis=1)
    else:
        X = base

    return X.astype(float)


print("Feature builder defined.")

Feature builder defined.


In [5]:
# ── Core two-step Lasso ──────────────────────────────────────────
def predict_income_for_role(
    df: pd.DataFrame,
    train_mask: pd.Series,
    prefix: str = "r",
    cv: int = 10,
    max_iter: int = 10_000
) -> pd.DataFrame:
    """
    Runs the two-step Lasso for one person role ('r' = respondent, 'sp' = spouse).

    Adds three columns to df:
      hat_pos_{prefix}      -- 0/1 predicted worker status (Step 1)
      hat_earn_{prefix}     -- predicted earnings level from Step 2 (exp of log model)
      hat_incearn_{prefix}  -- final income: Step-2 value if hat_pos=1, else 0
    """
    earn_col = f"{prefix}_incearn"

    # Rename spouse-prefixed columns to canonical 'r_*' names for the feature builder
    col_map = {
        "r_agegroup": f"{prefix}_agegroup",
        "r_edugroup": f"{prefix}_edugroup",
        "r_race":     f"{prefix}_race",
        "r_occbroad": f"{prefix}_occbroad",
        "r_degbroad": f"{prefix}_degbroad",
        "r_male":     f"{prefix}_male",
    }

    src_cols = ["statefip", "c_children", earn_col] + list(col_map.values())
    missing  = [c for c in src_cols if c not in df.columns]
    if missing:
        raise KeyError(f"predict_income_for_role(prefix='{prefix}'): "
                       f"missing columns {missing}")

    person = df[src_cols].copy().rename(
        columns={v: k for k, v in col_map.items()}
    )

    print(f"\n{'='*55}")
    print(f"  Role: {prefix} | N={len(person):,} | N_train={train_mask.sum():,}")

    # Feature matrix
    print("  Building features...")
    X_full = build_features(person)
    print(f"  Features: {X_full.shape[1]:,}")

    scaler  = StandardScaler()
    X_sc    = scaler.fit_transform(X_full)
    X_train = X_sc[train_mask.values]

    # ── Step 1: P(earn > 0) ──────────────────────────────────────
    print("  Step 1: LassoCV — P(incearn > 0)...")
    y1       = (person[earn_col] > 0).astype(float).values
    y1_train = y1[train_mask.values]

    m1 = LassoCV(cv=cv, max_iter=max_iter, n_jobs=-1, random_state=42)
    m1.fit(X_train, y1_train)

    hat_pos_prob = m1.predict(X_sc)
    threshold    = y1_train.mean()   # paper's threshold rule
    hat_pos      = (hat_pos_prob >= threshold).astype(float)

    r2_s1 = 1 - np.var(y1_train - m1.predict(X_train)) / np.var(y1_train)
    print(f"    alpha={m1.alpha_:.6f}  non-zero={np.sum(m1.coef_!=0)}  "
          f"threshold={threshold:.4f}  pred_share={hat_pos.mean():.4f}  "
          f"R²={r2_s1:.4f} (paper≈0.449)")

    # ── Step 2: log(earn) | earn > 0 ────────────────────────────
    print("  Step 2: LassoCV — log(incearn) | incearn > 0...")
    pos_train = train_mask.values & (person[earn_col].values > 0)
    y2_train  = np.log(person.loc[pos_train, earn_col].values)
    X2_train  = X_sc[pos_train]

    m2 = LassoCV(cv=cv, max_iter=max_iter, n_jobs=-1, random_state=42)
    m2.fit(X2_train, y2_train)

    hat_log_earn = m2.predict(X_sc)
    hat_earn_lvl = np.exp(hat_log_earn)

    r2_s2 = 1 - np.var(y2_train - m2.predict(X2_train)) / np.var(y2_train)
    print(f"    alpha={m2.alpha_:.6f}  non-zero={np.sum(m2.coef_!=0)}  "
          f"R²={r2_s2:.4f} (paper≈0.301)")

    # Final: zero non-workers
    hat_incearn = np.where(hat_pos == 1, hat_earn_lvl, 0.0)

    df = df.copy()
    df[f"hat_pos_{prefix}"]     = hat_pos
    df[f"hat_earn_{prefix}"]    = hat_earn_lvl
    df[f"hat_incearn_{prefix}"] = hat_incearn
    return df


print("Function defined.")

Function defined.


In [6]:
# ── Run — Respondent ────────────────────────────────────────────
train_mask = df["year"] == 2012
print(f"Training sample (2012): N={train_mask.sum():,}")

df = predict_income_for_role(df, train_mask, prefix="r")

Training sample (2012): N=5,070

  Role: r | N=37,907 | N_train=5,070
  Building features...
  Features: 906
  Step 1: LassoCV — P(incearn > 0)...
    alpha=0.010205  non-zero=31  threshold=0.8929  pred_share=0.4768  R²=0.0656 (paper≈0.449)
  Step 2: LassoCV — log(incearn) | incearn > 0...
    alpha=0.019921  non-zero=126  R²=0.2952 (paper≈0.301)


In [7]:
# ── Run — Spouse / partner ──────────────────────────────────────
df = predict_income_for_role(df, train_mask, prefix="sp")


  Role: sp | N=37,907 | N_train=5,070
  Building features...
  Features: 924
  Step 1: LassoCV — P(incearn > 0)...
    alpha=0.008141  non-zero=83  threshold=0.8562  pred_share=0.5554  R²=0.0877 (paper≈0.449)
  Step 2: LassoCV — log(incearn) | incearn > 0...
    alpha=0.019203  non-zero=141  R²=0.2440 (paper≈0.301)


In [8]:
# ── Couple-level aggregates ──────────────────────────────────────
df["hat_c_incearn"] = df["hat_incearn_r"] + df["hat_incearn_sp"]

# Primary earner share = max(r, sp) / total  (matches paper's Table 2)
total = df["hat_c_incearn"].replace(0, np.nan)
df["hat_c_split"] = (
    np.maximum(df["hat_incearn_r"], df["hat_incearn_sp"]) / total
).fillna(0.5)

# Also store the 5-polynomial terms — needed for IV income controls
for i in range(1, 6):
    df[f"hat_c_incearn{i}"] = (df["hat_c_incearn"] / 10_000) ** i
    df[f"hat_c_split{i}"]   = df["hat_c_split"] ** i

print("Couple-level columns added.")

Couple-level columns added.


In [9]:
# ── Diagnostics vs Table 2 ───────────────────────────────────────
print("="*60)
print("DIAGNOSTICS vs. Table 2")
print("="*60)

print("\n--- Share positive earnings ---")
print(f"  Respondent  reported={(df['r_incearn']>0).mean():.3f}  "
      f"predicted={df['hat_pos_r'].mean():.3f}  "
      f"(paper predicted: 0.963 mar / 0.969 coh)")
print(f"  Spouse      reported={(df['sp_incearn']>0).mean():.3f}  "
      f"predicted={df['hat_pos_sp'].mean():.3f}")

print("\n--- Couple predicted total earnings ---")
for label, mask in [("Married", df["sscouple_mar"]),
                    ("Cohabiting", df["sscouple_coh"])]:
    sub = df[mask]
    print(f"  {label}: {sub['hat_c_incearn'].mean():>10,.0f}  "
          f"(paper: mar≈110,729  coh≈102,953)")

print("\n--- Couple predicted earnings split ---")
for label, mask in [("Married", df["sscouple_mar"]),
                    ("Cohabiting", df["sscouple_coh"])]:
    sub = df[mask]
    pos = sub["hat_c_incearn"] > 0
    print(f"  {label}: {sub.loc[pos,'hat_c_split'].mean():.3f}  "
          f"(paper: mar≈0.648  coh≈0.641)")

print("\n--- Correlation reported vs predicted (positive earners) ---")
for pfx, label in [("r", "Respondent"), ("sp", "Spouse")]:
    both = (df[f"{pfx}_incearn"] > 0) & (df[f"hat_incearn_{pfx}"] > 0)
    r = np.corrcoef(df.loc[both, f"{pfx}_incearn"],
                    df.loc[both, f"hat_incearn_{pfx}"])[0, 1]
    print(f"  {label}: r={r:.3f}")

DIAGNOSTICS vs. Table 2

--- Share positive earnings ---
  Respondent  reported=0.905  predicted=0.477  (paper predicted: 0.963 mar / 0.969 coh)
  Spouse      reported=0.863  predicted=0.555

--- Couple predicted total earnings ---
  Married:     56,778  (paper: mar≈110,729  coh≈102,953)
  Cohabiting:     51,158  (paper: mar≈110,729  coh≈102,953)

--- Couple predicted earnings split ---
  Married: 0.783  (paper: mar≈0.648  coh≈0.641)
  Cohabiting: 0.800  (paper: mar≈0.648  coh≈0.641)

--- Correlation reported vs predicted (positive earners) ---
  Respondent: r=0.335
  Spouse: r=0.352


In [10]:
# ── Save ─────────────────────────────────────────────────────────
df.to_pickle(OUT_PKL)
print(f"Saved to {OUT_PKL}")
print(f"New columns: {[c for c in df.columns if c.startswith('hat_')]}")

Saved to /Users/trinityrose/Desktop/acs_ssc_predicted_v2.pkl
New columns: ['hat_pos_r', 'hat_earn_r', 'hat_incearn_r', 'hat_pos_sp', 'hat_earn_sp', 'hat_incearn_sp', 'hat_c_incearn', 'hat_c_split', 'hat_c_incearn1', 'hat_c_split1', 'hat_c_incearn2', 'hat_c_split2', 'hat_c_incearn3', 'hat_c_split3', 'hat_c_incearn4', 'hat_c_split4', 'hat_c_incearn5', 'hat_c_split5']


In [14]:
import json
with open("/Users/trinityrose/Desktop/Table A2 rep.ipynb") as f:
    nb = json.load(f)
src = "".join(nb["cells"][3]["source"])
print(src)

# Sort data to ensure merge logic is accurate
df_clean = df_full.sort_values(["year","serial","pernum"]).reset_index(drop=True)

# Extract potential partner info and merge it back to the main table
p_info = df_clean[["year","serial","pernum","sex","related","qrelate","marst"]].copy()
p_info.columns = ["year","serial","sploc","p_sex","p_related","p_qrelate","p_marst"]
df_clean = df_clean.merge(p_info, on=["year","serial","sploc"], how="left")

# Basic conditions: Same sex and valid spouse location (sploc)
same_sex = df_clean["sex"] == df_clean["p_sex"]
sploc_valid = df_clean["sploc"] > 0

# Identify same-sex married (ss_mar) and same-sex cohabiting (ss_coh)
ss_mar = same_sex & sploc_valid & (((df_clean["related"] == 101) & (df_clean["p_related"] == 201)) | ((df_clean["related"] == 201) & (df_clean["p_related"] == 101)))
ss_coh = same_sex & sploc_valid & (((df_clean["related"] == 101) & (df_clean["p_related"] == 1114)) | ((df_clean["related"] == 1114) & (df_clean["p_related"] == 101)))

